# Describe por categorias desde SQL

Este notebook consulta la tabla final `viabilidad_municipal` desde MySQL, une los clusters y genera `describe()` por `clasificacion_preliminar` para cada bloque de variables.

In [35]:
from pathlib import Path
import io
import subprocess
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path(r"C:\Users\tabo_\OneDrive\Desktop\JHON T\ITM\introduccion inteligencia artificial")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.db.mysql_cli import MySQLSettings, query_dataframe

settings = MySQLSettings.from_env()

def sql_df(sql: str) -> pd.DataFrame:
    try:
        return query_dataframe(sql, settings=settings, database=settings.database)
    except subprocess.CalledProcessError as exc:
        print("Error ejecutando SQL en MySQL:")
        print(exc.stderr or exc.output or exc)
        print("Consulta enviada:")
        print(sql)
        raise

def existing_columns(df: pd.DataFrame, columns: list[str]) -> list[str]:
    return [column for column in columns if column in df.columns]

def dataframe_info(df: pd.DataFrame) -> str:
    buffer = io.StringIO()
    df.info(buf=buffer)
    return buffer.getvalue()

def null_report(df: pd.DataFrame, treatment: dict[str, str] | None = None) -> pd.DataFrame:
    treatment = treatment or {}
    report = pd.DataFrame({
        "columna": df.columns,
        "nulos": [int(df[column].isna().sum()) for column in df.columns],
        "pct_nulos": [round(float(df[column].isna().mean() * 100), 2) for column in df.columns],
        "tratamiento": [treatment.get(column, "Sin nulos o se conserva como dato valido") for column in df.columns],
    })
    return report.sort_values(["nulos", "columna"], ascending=[False, True])

def inspect_source(df: pd.DataFrame, treatment: dict[str, str] | None = None, numeric_cols: list[str] | None = None) -> None:
    print(f"Estructura: {len(df):,} filas x {len(df.columns):,} columnas")
    print("\n.info()")
    print(dataframe_info(df))
    display(df.head())
    if numeric_cols is None:
        display(df.describe(include="all").T)
    else:
        display(df[existing_columns(df, numeric_cols)].describe().T)
    display(null_report(df, treatment))

def describe_by_category(df: pd.DataFrame, columns: list[str], category: str = "clasificacion_preliminar") -> pd.DataFrame:
    cols = existing_columns(df, [category, *columns])
    work = df[cols].copy()
    numeric_cols = [column for column in cols if column != category]
    for column in numeric_cols:
        work[column] = pd.to_numeric(work[column], errors="coerce")
    return work.groupby(category, dropna=False)[numeric_cols].describe().T

print(f"Base MySQL: {settings.database}")

Base MySQL: granja_solar


In [36]:
tabla_final = sql_df("""
SELECT
    v.*,
    c.cluster_kmeans,
    c.cluster_kmeans_label,
    c.silhouette_municipio
FROM viabilidad_municipal v
LEFT JOIN cluster_municipal c
    ON c.codigo_dane = v.codigo_dane;
""")

print(f"Filas: {len(tabla_final):,}")
print(f"Columnas: {len(tabla_final.columns):,}")
display(tabla_final.head())
display(tabla_final["clasificacion_preliminar"].value_counts(dropna=False).rename_axis("clasificacion").reset_index(name="municipios"))

Filas: 1,104
Columnas: 77


,codigo_dane,municipio,departamento,categoria_nombre,area_km2_igac,altitud_m,lon,lat,pvout_kwh_kwp_day,annual_yield_kwh_kw_year,...,bono_demanda_favorable,score_rural_con_bono_demanda,v_i_modelo_proxy_xm,clasificacion_preliminar,v_i_modelo_oficial,estado_modelo_oficial,notas_metodologicas,cluster_kmeans,cluster_kmeans_label,silhouette_municipio
0,5001,Medellín,Antioquia,Distrito,373.440416,1475,-75.602895,6.269145,4.371,1595.414932,...,0.0,0.0,0.0,excluida_preliminar,0.0,modelo_rural_sin_demanda_con_pot_pendiente: no...,"Score preliminar usa PVOUT puntual municipal, ...",NaN,NaN,NaN
1,5002,Abejorral,Antioquia,Municipio,506.952798,2275,-75.429630,5.805339,4.286,1564.389918,...,0.0,0.0,0.0,excluida_preliminar,0.0,modelo_rural_sin_demanda_con_pot_pendiente: no...,"Score preliminar usa PVOUT puntual municipal, ...",NaN,NaN,NaN
2,5004,Abriaquí,Antioquia,Municipio,296.974192,1900,-76.083417,6.629176,3.780,1379.699990,...,0.0,0.0,0.0,excluida_preliminar,0.0,modelo_rural_sin_demanda_con_pot_pendiente: no...,"Score preliminar usa PVOUT puntual municipal, ...",NaN,NaN,NaN
3,5021,Alejandría,Antioquia,Municipio,128.856440,1750,-75.099281,6.356241,4.103,1497.595060,...,0.0,0.0,0.0,excluida_preliminar,0.0,modelo_rural_sin_demanda_con_pot_pendiente: no...,"Score preliminar usa PVOUT puntual municipal, ...",NaN,NaN,NaN
4,5030,Amagá,Antioquia,Municipio,84.118977,1400,-75.703795,6.034324,4.134,1508.909936,...,0.0,0.0,0.0,excluida_preliminar,0.0,modelo_rural_sin_demanda_con_pot_pendiente: no...,"Score preliminar usa PVOUT puntual municipal, ...",NaN,NaN,NaN


,clasificacion,municipios
0,excluida_preliminar,699
1,baja_preliminar,202
2,media_preliminar,101
3,alta_preliminar,61
4,muy_alta_preliminar,41


In [37]:
# Fuente 1: PVOUT Global Solar Atlas - puntos municipales
pvout = sql_df("""
SELECT
    pv.codigo_dane,
    pv.pvout_kwh_kwp_day,
    pv.annual_yield_kwh_kw_year,
    m.lat,
    m.lon
FROM municipio_pvout pv
LEFT JOIN municipios m
    ON m.codigo_dane = pv.codigo_dane;
""")

pvout_treatment = {
    "codigo_dane": "0 nulos esperados; se normaliza a CHAR(5) y queda como llave primaria.",
    "pvout_kwh_kwp_day": "0 nulos esperados; se mantiene como variable solar base.",
    "annual_yield_kwh_kw_year": "0 nulos esperados; se deriva de PVOUT diario y se mantiene.",
    "lat": "0 nulos esperados; coordenada de muestreo municipal.",
    "lon": "0 nulos esperados; coordenada de muestreo municipal.",
}

inspect_source(pvout, pvout_treatment, ["pvout_kwh_kwp_day", "annual_yield_kwh_kw_year", "lat", "lon"])
display(pvout["codigo_dane"].duplicated().sum())
print("Interpretacion: la distribucion de PVOUT se revisa como recurso solar, pero no decide sola la viabilidad final.")

Estructura: 1,104 filas x 5 columnas

.info()
<class 'pandas.DataFrame'>
RangeIndex: 1104 entries, 0 to 1103
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   codigo_dane               1104 non-null   int64  
 1   pvout_kwh_kwp_day         1104 non-null   float64
 2   annual_yield_kwh_kw_year  1104 non-null   float64
 3   lat                       1104 non-null   float64
 4   lon                       1104 non-null   float64
dtypes: float64(4), int64(1)
memory usage: 43.3 KB



,codigo_dane,pvout_kwh_kwp_day,annual_yield_kwh_kw_year,lat,lon
0,5001,4.371,1595.414932,6.269145,-75.602895
1,5002,4.286,1564.389918,5.805339,-75.429630
2,5004,3.780,1379.699990,6.629176,-76.083417
3,5021,4.103,1497.595060,6.356241,-75.099281
4,5030,4.134,1508.909936,6.034324,-75.703795


,count,mean,std,min,25%,50%,75%,max
pvout_kwh_kwp_day,1104.0,3.944247,0.459812,2.468000,3.649500,4.005000,4.299500,4.926000
annual_yield_kwh_kw_year,1104.0,1439.650259,167.831493,900.819976,1332.067505,1461.825042,1569.317452,1797.990043
lat,1104.0,5.695586,2.566333,-3.610539,4.312348,5.574426,7.055209,13.353211
lon,1104.0,-74.714532,1.575921,-81.720352,-75.788747,-74.774613,-73.496989,-68.218951


,columna,nulos,pct_nulos,tratamiento
2,annual_yield_kwh_kw_year,0,0.0,0 nulos esperados; se deriva de PVOUT diario y...
0,codigo_dane,0,0.0,0 nulos esperados; se normaliza a CHAR(5) y qu...
3,lat,0,0.0,0 nulos esperados; coordenada de muestreo muni...
4,lon,0,0.0,0 nulos esperados; coordenada de muestreo muni...
1,pvout_kwh_kwp_day,0,0.0,0 nulos esperados; se mantiene como variable s...


np.int64(0)

Interpretacion: la distribucion de PVOUT se revisa como recurso solar, pero no decide sola la viabilidad final.


In [38]:
# Fuente 2: Pendientes IGAC - muestreo puntual
pendientes = sql_df("""
SELECT *
FROM municipio_pendiente;
""")

pendientes_treatment = {
    "codigo_dane": "0 nulos esperados; llave municipal normalizada a CHAR(5).",
    "pendiente_igac": "Si no hubo respuesta IGAC, se marca como desconocida o no viable segun pipeline.",
    "viabilidad_pendiente": "Nulos/sin respuesta se tratan como exclusion fisica en el score final.",
    "score_pendiente": "Sin respuesta se lleva a 0 o queda excluida por R_i segun la tabla final.",
    "p_i_pendiente_proxy": "Proxy final: pendiente favorable mantiene valor alto; empinado/sin dato penaliza.",
}

inspect_source(pendientes, pendientes_treatment)
if "viabilidad_pendiente" in pendientes.columns:
    display(pendientes["viabilidad_pendiente"].value_counts(dropna=False).rename_axis("viabilidad_pendiente").reset_index(name="municipios"))
print("Interpretacion: pendiente opera como filtro fisico restrictivo; clases empinadas o sin respuesta reducen o anulan viabilidad.")

Estructura: 1,104 filas x 15 columnas

.info()
<class 'pandas.DataFrame'>
RangeIndex: 1104 entries, 0 to 1103
Data columns (total 15 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   codigo_dane                    1104 non-null   int64  
 1   indice_punto                   1104 non-null   int64  
 2   lon_consulta                   1104 non-null   float64
 3   lat_consulta                   1104 non-null   float64
 4   pendiente_igac                 1102 non-null   str    
 5   identify_ok                    1104 non-null   int64  
 6   identify_mensaje               2 non-null      str    
 7   clase_igac                     1102 non-null   str    
 8   pendiente_min_pct              1102 non-null   float64
 9   pendiente_max_pct              469 non-null    float64
 10  max_slope_percent              1104 non-null   int64  
 11  conditional_max_slope_percent  1104 non-null   int64  
 12  viabilidad_p

,codigo_dane,indice_punto,lon_consulta,lat_consulta,pendiente_igac,identify_ok,identify_mensaje,clase_igac,pendiente_min_pct,pendiente_max_pct,max_slope_percent,conditional_max_slope_percent,viabilidad_pendiente,score_pendiente,criterio
0,5001,70,-75.602895,6.269145,Empinado (>14%),1,NaN,Empinado (>14%),14.0,NaN,7,14,no_viable,0.0,Clase abierta desde 14.0%; se excluye por prec...
1,5002,2,-75.429630,5.805339,Empinado (>14%),1,NaN,Empinado (>14%),14.0,NaN,7,14,no_viable,0.0,Clase abierta desde 14.0%; se excluye por prec...
2,5004,3,-76.083417,6.629176,Empinado (>14%),1,NaN,Empinado (>14%),14.0,NaN,7,14,no_viable,0.0,Clase abierta desde 14.0%; se excluye por prec...
3,5021,4,-75.099281,6.356241,Empinado (>14%),1,NaN,Empinado (>14%),14.0,NaN,7,14,no_viable,0.0,Clase abierta desde 14.0%; se excluye por prec...
4,5030,5,-75.703795,6.034324,Empinado (>14%),1,NaN,Empinado (>14%),14.0,NaN,7,14,no_viable,0.0,Clase abierta desde 14.0%; se excluye por prec...


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
codigo_dane,1104.0,NaN,NaN,NaN,37759.651268,25782.638439,5001.0,15656.5,25789.0,63418.25,99773.0
indice_punto,1104.0,NaN,NaN,NaN,551.5,318.841653,0.0,275.75,551.5,827.25,1103.0
lon_consulta,1104.0,NaN,NaN,NaN,-74.714532,1.575921,-81.720352,-75.788747,-74.774613,-73.496989,-68.218951
lat_consulta,1104.0,NaN,NaN,NaN,5.695586,2.566333,-3.610539,4.312348,5.574426,7.055209,13.353211
pendiente_igac,1102,3,Empinado (>14%),633,NaN,NaN,NaN,NaN,NaN,NaN,NaN
identify_ok,1104.0,NaN,NaN,NaN,0.998188,0.042544,0.0,1.0,1.0,1.0,1.0
identify_mensaje,2,1,Sin resultado en IGAC para el punto.,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
clase_igac,1102,3,Empinado (>14%),633,NaN,NaN,NaN,NaN,NaN,NaN,NaN
pendiente_min_pct,1102.0,NaN,NaN,NaN,9.064428,6.132982,0.0,0.0,14.0,14.0,14.0
pendiente_max_pct,469.0,NaN,NaN,NaN,9.402985,3.327185,7.0,7.0,7.0,14.0,14.0


,columna,nulos,pct_nulos,tratamiento
6,identify_mensaje,1102,99.82,Sin nulos o se conserva como dato valido
9,pendiente_max_pct,635,57.52,Sin nulos o se conserva como dato valido
7,clase_igac,2,0.18,Sin nulos o se conserva como dato valido
4,pendiente_igac,2,0.18,"Si no hubo respuesta IGAC, se marca como desco..."
8,pendiente_min_pct,2,0.18,Sin nulos o se conserva como dato valido
13,score_pendiente,2,0.18,Sin respuesta se lleva a 0 o queda excluida po...
0,codigo_dane,0,0.00,0 nulos esperados; llave municipal normalizada...
11,conditional_max_slope_percent,0,0.00,Sin nulos o se conserva como dato valido
14,criterio,0,0.00,Sin nulos o se conserva como dato valido
5,identify_ok,0,0.00,Sin nulos o se conserva como dato valido


,viabilidad_pendiente,municipios
0,no_viable,633
1,viable,308
2,condicional,161
3,desconocida,2


Interpretacion: pendiente opera como filtro fisico restrictivo; clases empinadas o sin respuesta reducen o anulan viabilidad.


In [39]:
# Fuente 3: Subestaciones UPME y distancia a red
red = sql_df("""
SELECT *
FROM municipio_red;
""")

red_treatment = {
    "codigo_dane": "0 nulos esperados; llave municipal normalizada a CHAR(5).",
    "dist_subestacion_km": "Si aparece nulo, se imputa con la mediana solo para graficos exploratorios; el municipio ya queda penalizado/excluido en el score.",
    "id_subestacion_mas_cercana": "Nulos indican que no se encontro subestacion cercana usable; se conserva para auditoria.",
    "subestacion_mas_cercana": "Nulos se conservan para auditoria; no se inventa nombre de subestacion.",
}

inspect_source(red, red_treatment)
red_graficos = red.copy()
if "dist_subestacion_km" in red_graficos.columns:
    mediana_distancia = red_graficos["dist_subestacion_km"].median()
    red_graficos["dist_subestacion_km_imputada_graficos"] = red_graficos["dist_subestacion_km"].fillna(mediana_distancia)
    print(f"Mediana usada solo para graficos exploratorios: {mediana_distancia:.2f} km")
    display(red_graficos[["dist_subestacion_km", "dist_subestacion_km_imputada_graficos"]].describe().T)
print("Interpretacion: la distancia a red discrimina municipios con conexion mas viable y penaliza zonas aisladas.")

Estructura: 1,104 filas x 12 columnas

.info()
<class 'pandas.DataFrame'>
RangeIndex: 1104 entries, 0 to 1103
Data columns (total 12 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   codigo_dane                     1104 non-null   int64  
 1   dist_subestacion_km             1104 non-null   float64
 2   subestacion_mas_cercana         1104 non-null   str    
 3   id_subestacion_mas_cercana      1104 non-null   int64  
 4   cod_sub_upme_mas_cercana        1104 non-null   int64  
 5   nivel_tension_mas_cercana       1104 non-null   str    
 6   tension_mas_cercana             1104 non-null   str    
 7   estado_subestacion_mas_cercana  1104 non-null   str    
 8   capacidad_mva_mas_cercana       1104 non-null   float64
 9   subestacion_lon                 1104 non-null   float64
 10  subestacion_lat                 1104 non-null   float64
 11  criterio_red                    1104 non-null   str    
dty

,codigo_dane,dist_subestacion_km,subestacion_mas_cercana,id_subestacion_mas_cercana,cod_sub_upme_mas_cercana,nivel_tension_mas_cercana,tension_mas_cercana,estado_subestacion_mas_cercana,capacidad_mva_mas_cercana,subestacion_lon,subestacion_lat,criterio_red
0,5001,0.295315,COLOMBIA,635,635,"Tensión nominal mayor o igual a 57,5 kV y meno...","110/44/13,2",En Servicio,120.0,-75.6037,6.2666,Distancia euclidiana en EPSG:3116 desde punto ...
1,5002,25.910980,LA CEJA,665,665,"Tensión nominal mayor o igual a 57,5 kV y meno...","110/44/13,2",En Servicio,60.0,-75.4163,6.0392,Distancia euclidiana en EPSG:3116 desde punto ...
2,5004,23.355306,CHORODO,651,651,"Tensión nominal mayor o igual a 57,5 kV y meno...","110/44/13,2",En Servicio,45.0,-76.1250,6.8361,Distancia euclidiana en EPSG:3116 desde punto ...
3,5021,9.113458,GUATAPE,656,656,"Tensión nominal mayor o igual a 57,5 kV y meno...","110/44/13,2",En Servicio,25.0,-75.1023,6.2739,Distancia euclidiana en EPSG:3116 desde punto ...
4,5030,2.979491,AMAGA,606,606,"Tensión nominal mayor o igual a 57,5 kV y meno...","110/44/13,2",En Servicio,40.0,-75.6929,6.0097,Distancia euclidiana en EPSG:3116 desde punto ...


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
codigo_dane,1104.0,NaN,NaN,NaN,37759.651268,25782.638439,5001.0,15656.5,25789.0,63418.25,99773.0
dist_subestacion_km,1104.0,NaN,NaN,NaN,29.102714,53.824161,0.295315,10.977208,18.632091,31.138573,737.947838
subestacion_mas_cercana,1104,252,BOAVITA,27,NaN,NaN,NaN,NaN,NaN,NaN,NaN
id_subestacion_mas_cercana,1104.0,NaN,NaN,NaN,947.666667,500.35336,18.0,585.5,963.0,1390.0,1657.0
cod_sub_upme_mas_cercana,1104.0,NaN,NaN,NaN,947.666667,500.35336,18.0,585.5,963.0,1390.0,1657.0
nivel_tension_mas_cercana,1104,2,"Tensión nominal mayor o igual a 57,5 kV y meno...",1067,NaN,NaN,NaN,NaN,NaN,NaN,NaN
tension_mas_cercana,1104,20,"115/34,5",291,NaN,NaN,NaN,NaN,NaN,NaN,NaN
estado_subestacion_mas_cercana,1104,1,En Servicio,1104,NaN,NaN,NaN,NaN,NaN,NaN,NaN
capacidad_mva_mas_cercana,1104.0,NaN,NaN,NaN,49.054063,54.806423,0.01,24.75,40.0,60.0,900.0
subestacion_lon,1104.0,NaN,NaN,NaN,-74.730762,1.457638,-78.7669,-75.787737,-74.838669,-73.442983,-71.614279


,columna,nulos,pct_nulos,tratamiento
8,capacidad_mva_mas_cercana,0,0.0,Sin nulos o se conserva como dato valido
4,cod_sub_upme_mas_cercana,0,0.0,Sin nulos o se conserva como dato valido
0,codigo_dane,0,0.0,0 nulos esperados; llave municipal normalizada...
11,criterio_red,0,0.0,Sin nulos o se conserva como dato valido
1,dist_subestacion_km,0,0.0,"Si aparece nulo, se imputa con la mediana solo..."
7,estado_subestacion_mas_cercana,0,0.0,Sin nulos o se conserva como dato valido
3,id_subestacion_mas_cercana,0,0.0,Nulos indican que no se encontro subestacion c...
5,nivel_tension_mas_cercana,0,0.0,Sin nulos o se conserva como dato valido
10,subestacion_lat,0,0.0,Sin nulos o se conserva como dato valido
9,subestacion_lon,0,0.0,Sin nulos o se conserva como dato valido


Mediana usada solo para graficos exploratorios: 18.63 km


,count,mean,std,min,25%,50%,75%,max
dist_subestacion_km,1104.0,29.102714,53.824161,0.295315,10.977208,18.632091,31.138573,737.947838
dist_subestacion_km_imputada_graficos,1104.0,29.102714,53.824161,0.295315,10.977208,18.632091,31.138573,737.947838


Interpretacion: la distancia a red discrimina municipios con conexion mas viable y penaliza zonas aisladas.


In [40]:
# Fuente 4: RUNAP - areas protegidas
runap = sql_df("""
SELECT *
FROM municipio_runap;
""")

runap_treatment = {
    "codigo_dane": "0 nulos esperados; llave municipal normalizada a CHAR(5).",
    "pct_area_protegida_runap": "Nulo se interpreta como 0 interseccion cuando el pipeline no encontro poligono RUNAP.",
    "u_i_no_protegido_runap": "Se deriva como proporcion no protegida; si RUNAP cubre demasiado, baja el score territorial.",
    "r_i_runap": "Coberturas muy altas activan restriccion dura R_i=0.",
    "clasificacion_restriccion_runap": "Nulos se conservan o se etiquetan como sin_interseccion segun origen.",
}

inspect_source(runap, runap_treatment)
if "clasificacion_restriccion_runap" in runap.columns:
    display(runap["clasificacion_restriccion_runap"].value_counts(dropna=False).rename_axis("restriccion_runap").reset_index(name="municipios"))
print("Interpretacion: RUNAP no penaliza por existir interseccion, penaliza cuando la cobertura protegida domina el municipio.")

Estructura: 1,104 filas x 11 columnas

.info()
<class 'pandas.DataFrame'>
RangeIndex: 1104 entries, 0 to 1103
Data columns (total 11 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   codigo_dane                      1104 non-null   int64  
 1   area_municipio_km2_calc          1104 non-null   float64
 2   area_protegida_km2_runap         1104 non-null   float64
 3   pct_area_protegida_runap_raw     1104 non-null   float64
 4   pct_area_protegida_runap         1104 non-null   float64
 5   area_no_protegida_km2_runap      1104 non-null   float64
 6   u_i_no_protegido_runap           1104 non-null   float64
 7   r_i_runap                        1104 non-null   int64  
 8   umbral_exclusion_runap           1104 non-null   float64
 9   clasificacion_restriccion_runap  1104 non-null   str    
 10  criterio_runap                   1104 non-null   str    
dtypes: float64(7), int64(2), str(2)
memory usage: 

,codigo_dane,area_municipio_km2_calc,area_protegida_km2_runap,pct_area_protegida_runap_raw,pct_area_protegida_runap,area_no_protegida_km2_runap,u_i_no_protegido_runap,r_i_runap,umbral_exclusion_runap,clasificacion_restriccion_runap,criterio_runap
0,5001,373.532556,166.161709,0.444839,0.444839,207.370846,0.555161,1,0.8,restriccion_alta,R_i_RUNAP=0 si el area protegida RUNAP cubre a...
1,5002,507.134108,17.831689,0.035162,0.035162,489.302419,0.964838,1,0.8,restriccion_baja_media,R_i_RUNAP=0 si el area protegida RUNAP cubre a...
2,5004,296.955984,12.232858,0.041194,0.041194,284.723126,0.958806,1,0.8,restriccion_baja_media,R_i_RUNAP=0 si el area protegida RUNAP cubre a...
3,5021,128.932162,6.797092,0.052718,0.052718,122.135070,0.947282,1,0.8,restriccion_baja_media,R_i_RUNAP=0 si el area protegida RUNAP cubre a...
4,5030,84.134310,5.011393,0.059564,0.059564,79.122916,0.940436,1,0.8,restriccion_baja_media,R_i_RUNAP=0 si el area protegida RUNAP cubre a...


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
codigo_dane,1104.0,NaN,NaN,NaN,37759.651268,25782.638439,5001.0,15656.5,25789.0,63418.25,99773.0
area_municipio_km2_calc,1104.0,NaN,NaN,NaN,880.981428,2988.502022,15.736751,130.958585,279.568209,642.433955,65465.653706
area_protegida_km2_runap,1104.0,NaN,NaN,NaN,141.017646,859.60938,0.0,0.0,3.011479,54.143549,22162.845068
pct_area_protegida_runap_raw,1104.0,NaN,NaN,NaN,0.114554,0.190047,0.0,0.0,0.00918,0.161201,1.0
pct_area_protegida_runap,1104.0,NaN,NaN,NaN,0.114554,0.190047,0.0,0.0,0.00918,0.161201,1.0
area_no_protegida_km2_runap,1104.0,NaN,NaN,NaN,739.963782,2484.06961,0.0,112.725953,237.369851,559.66662,59941.976714
u_i_no_protegido_runap,1104.0,NaN,NaN,NaN,0.885446,0.190047,0.0,0.838799,0.99082,1.0,1.0
r_i_runap,1104.0,NaN,NaN,NaN,0.986413,0.115821,0.0,1.0,1.0,1.0,1.0
umbral_exclusion_runap,1104.0,NaN,NaN,NaN,0.8,0.0,0.8,0.8,0.8,0.8,0.8
clasificacion_restriccion_runap,1104,4,restriccion_baja_media,602,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,columna,nulos,pct_nulos,tratamiento
1,area_municipio_km2_calc,0,0.0,Sin nulos o se conserva como dato valido
5,area_no_protegida_km2_runap,0,0.0,Sin nulos o se conserva como dato valido
2,area_protegida_km2_runap,0,0.0,Sin nulos o se conserva como dato valido
9,clasificacion_restriccion_runap,0,0.0,Nulos se conservan o se etiquetan como sin_int...
0,codigo_dane,0,0.0,0 nulos esperados; llave municipal normalizada...
10,criterio_runap,0,0.0,Sin nulos o se conserva como dato valido
4,pct_area_protegida_runap,0,0.0,Nulo se interpreta como 0 interseccion cuando ...
3,pct_area_protegida_runap_raw,0,0.0,Sin nulos o se conserva como dato valido
7,r_i_runap,0,0.0,Coberturas muy altas activan restriccion dura ...
6,u_i_no_protegido_runap,0,0.0,Se deriva como proporcion no protegida; si RUN...


,restriccion_runap,municipios
0,restriccion_baja_media,602
1,sin_area_protegida,341
2,restriccion_alta,146
3,exclusion_preliminar,15


Interpretacion: RUNAP no penaliza por existir interseccion, penaliza cuando la cobertura protegida domina el municipio.


In [41]:
# 3.3 Integracion de fuentes: tabla final categorizada desde SQL
integracion_treatment = {
    "codigo_dane": "Llave primaria municipal normalizada a 5 digitos; no se imputan codigos.",
    "demanda_xm_proxy_mwh_o_unidad_fuente": "Los nulos de demanda se mantienen porque la demanda es contexto, no criterio principal del ranking rural.",
    "d_i_demanda": "Los nulos se mantienen o quedan como 0/NaN segun disponibilidad; no reemplazan el score rural.",
    "dist_subestacion_km": "Nulos se pueden imputar con mediana solo para graficos exploratorios; en el modelo quedan penalizados o excluidos.",
    "pvout_kwh_kwp_day": "No se imputan valores solares; si faltara PVOUT se debe corregir fuente antes del scoring.",
    "pendiente_igac": "Sin respuesta de pendiente queda como desconocida/no viable y afecta R_i o P_i.",
    "pct_area_protegida_runap": "Si no hay interseccion RUNAP se trata como 0 cobertura protegida.",
    "clasificacion_preliminar": "Categoria final derivada del score; no se imputa.",
}

inspect_source(tabla_final, integracion_treatment)
display(tabla_final["clasificacion_preliminar"].value_counts(dropna=False).rename_axis("clasificacion_preliminar").reset_index(name="municipios"))

columnas_revision = existing_columns(tabla_final, [
    "codigo_dane", "pvout_kwh_kwp_day", "pendiente_igac", "dist_subestacion_km",
    "pct_area_protegida_runap", "demanda_xm_proxy_mwh_o_unidad_fuente",
    "v_i_modelo_rural", "clasificacion_preliminar"
])
display(null_report(tabla_final[columnas_revision], integracion_treatment))
print("Resultado esperado: base municipal estricta con uniones left join 1:1 por codigo_dane y categorias finales para analisis.")

Estructura: 1,104 filas x 77 columnas

.info()
<class 'pandas.DataFrame'>
RangeIndex: 1104 entries, 0 to 1103
Data columns (total 77 columns):
 #   Column                                      Non-Null Count  Dtype  
---  ------                                      --------------  -----  
 0   codigo_dane                                 1104 non-null   int64  
 1   municipio                                   1104 non-null   str    
 2   departamento                                1104 non-null   str    
 3   categoria_nombre                            1104 non-null   str    
 4   area_km2_igac                               1104 non-null   float64
 5   altitud_m                                   1104 non-null   int64  
 6   lon                                         1104 non-null   float64
 7   lat                                         1104 non-null   float64
 8   pvout_kwh_kwp_day                           1104 non-null   float64
 9   annual_yield_kwh_kw_year                    1104 

,codigo_dane,municipio,departamento,categoria_nombre,area_km2_igac,altitud_m,lon,lat,pvout_kwh_kwp_day,annual_yield_kwh_kw_year,...,bono_demanda_favorable,score_rural_con_bono_demanda,v_i_modelo_proxy_xm,clasificacion_preliminar,v_i_modelo_oficial,estado_modelo_oficial,notas_metodologicas,cluster_kmeans,cluster_kmeans_label,silhouette_municipio
0,5001,Medellín,Antioquia,Distrito,373.440416,1475,-75.602895,6.269145,4.371,1595.414932,...,0.0,0.0,0.0,excluida_preliminar,0.0,modelo_rural_sin_demanda_con_pot_pendiente: no...,"Score preliminar usa PVOUT puntual municipal, ...",NaN,NaN,NaN
1,5002,Abejorral,Antioquia,Municipio,506.952798,2275,-75.429630,5.805339,4.286,1564.389918,...,0.0,0.0,0.0,excluida_preliminar,0.0,modelo_rural_sin_demanda_con_pot_pendiente: no...,"Score preliminar usa PVOUT puntual municipal, ...",NaN,NaN,NaN
2,5004,Abriaquí,Antioquia,Municipio,296.974192,1900,-76.083417,6.629176,3.780,1379.699990,...,0.0,0.0,0.0,excluida_preliminar,0.0,modelo_rural_sin_demanda_con_pot_pendiente: no...,"Score preliminar usa PVOUT puntual municipal, ...",NaN,NaN,NaN
3,5021,Alejandría,Antioquia,Municipio,128.856440,1750,-75.099281,6.356241,4.103,1497.595060,...,0.0,0.0,0.0,excluida_preliminar,0.0,modelo_rural_sin_demanda_con_pot_pendiente: no...,"Score preliminar usa PVOUT puntual municipal, ...",NaN,NaN,NaN
4,5030,Amagá,Antioquia,Municipio,84.118977,1400,-75.703795,6.034324,4.134,1508.909936,...,0.0,0.0,0.0,excluida_preliminar,0.0,modelo_rural_sin_demanda_con_pot_pendiente: no...,"Score preliminar usa PVOUT puntual municipal, ...",NaN,NaN,NaN


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
codigo_dane,1104.0,NaN,NaN,NaN,37759.651268,25782.638439,5001.0,15656.5,25789.0,63418.25,99773.0
municipio,1104,1020,La Unión,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
departamento,1104,32,Antioquia,125,NaN,NaN,NaN,NaN,NaN,NaN,NaN
categoria_nombre,1104,2,Municipio,1092,NaN,NaN,NaN,NaN,NaN,NaN,NaN
area_km2_igac,1104.0,NaN,NaN,NaN,879.538032,2978.924623,15.732852,130.882807,279.439405,642.022539,65187.505271
...,...,...,...,...,...,...,...,...,...,...,...
estado_modelo_oficial,1104,4,modelo_rural_sin_demanda_con_pot_pendiente: no...,816,NaN,NaN,NaN,NaN,NaN,NaN,NaN
notas_metodologicas,1104,1,"Score preliminar usa PVOUT puntual municipal, ...",1104,NaN,NaN,NaN,NaN,NaN,NaN,NaN
cluster_kmeans,405.0,NaN,NaN,NaN,0.362963,0.481449,0.0,0.0,0.0,1.0,1.0
cluster_kmeans_label,405,2,cluster_0,258,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,columna,nulos,pct_nulos,tratamiento
14,identify_mensaje,1102,99.82,Sin nulos o se conserva como dato valido
39,apto_doble_uso_pastoreo,1101,99.73,Sin nulos o se conserva como dato valido
31,layer_id_pot,1101,99.73,Sin nulos o se conserva como dato valido
32,layer_name_pot,1101,99.73,Sin nulos o se conserva como dato valido
44,municipio_pot,1101,99.73,Sin nulos o se conserva como dato valido
...,...,...,...,...
61,u_i_uso_suelo,0,0.00,Sin nulos o se conserva como dato valido
71,v_i_modelo_oficial,0,0.00,Sin nulos o se conserva como dato valido
69,v_i_modelo_proxy_xm,0,0.00,Sin nulos o se conserva como dato valido
65,v_i_modelo_rural,0,0.00,Sin nulos o se conserva como dato valido


,clasificacion_preliminar,municipios
0,excluida_preliminar,699
1,baja_preliminar,202
2,media_preliminar,101
3,alta_preliminar,61
4,muy_alta_preliminar,41


,columna,nulos,pct_nulos,tratamiento
5,demanda_xm_proxy_mwh_o_unidad_fuente,48,4.35,Los nulos de demanda se mantienen porque la de...
2,pendiente_igac,2,0.18,Sin respuesta de pendiente queda como desconoc...
7,clasificacion_preliminar,0,0.00,Categoria final derivada del score; no se imputa.
0,codigo_dane,0,0.00,Llave primaria municipal normalizada a 5 digit...
3,dist_subestacion_km,0,0.00,Nulos se pueden imputar con mediana solo para ...
4,pct_area_protegida_runap,0,0.00,Si no hay interseccion RUNAP se trata como 0 c...
1,pvout_kwh_kwp_day,0,0.00,No se imputan valores solares; si faltara PVOU...
6,v_i_modelo_rural,0,0.00,Sin nulos o se conserva como dato valido


Resultado esperado: base municipal estricta con uniones left join 1:1 por codigo_dane y categorias finales para analisis.


In [42]:
# Describe: variables de identificacion y territorio
identificacion_cols = [
    "area_km2_igac",
    "lat",
    "lon",
]

display(describe_by_category(tabla_final, identificacion_cols))

clasificacion_preliminar  alta_preliminar  baja_preliminar  \
area_km2_igac count             61.000000       202.000000   
              mean             510.363922       630.046428   
              std              533.251661      1351.107705   
              min               42.058087        19.507853   
              25%              149.437700       118.806773   
              50%              361.188473       294.619284   
              75%              567.136096       763.675486   
              max             3082.410947     17199.254178   
lat           count             61.000000       202.000000   
              mean               8.123508         5.476164   
              std                2.569031         2.486901   
              min                3.052571         0.311752   
              25%                5.514772         4.118901   
              50%                9.245790         5.418847   
              75%               10.243497         6.977941   
              max               11.235309        10.978284   
lon           count             61.000000       202.000000   
              mean             -74.738430       -74.953005   
              std                0.782666         1.450223   
              min              -76.630227       -78.690693   
              25%              -75.115206       -75.945248   
              50%              -74.806730       -75.041943   
              75%              -74.253222       -73.625439   
              max              -72.882516       -71.592811   

clasificacion_preliminar  excluida_preliminar  media_preliminar  \
area_km2_igac count                699.000000        101.000000   
              mean                1030.594789        730.118721   
              std                 3636.146556       1041.306896   
              min                   15.732852         46.313558   
              25%                  132.295172        168.026174   
              50%                  262.704700        403.013859   
              75%                  592.173468        834.583440   
              max                65187.505271       6898.196766   
lat           count                699.000000        101.000000   
              mean                   5.203125          6.594308   
              std                    2.281695          2.588624   
              min                   -3.610539          1.806067   
              25%                    4.167660          4.372570   
              50%                    5.391299          6.059510   
              75%                    6.454900          9.021399   
              max                   13.353211         11.244923   
lon           count                699.000000        101.000000   
              mean                 -74.689756        -74.567153   
              std                    1.731081          1.174228   
              min                  -81.720352        -77.162790   
              25%                  -75.922895        -75.376767   
              50%                  -74.682733        -74.735963   
              75%                  -73.318228        -73.772925   
              max                  -68.218951        -71.583196   

clasificacion_preliminar  muy_alta_preliminar  
area_km2_igac count                 41.000000  
              mean                 450.747846  
              std                  712.655677  
              min                   42.246025  
              25%                  109.395002  
              50%                  249.094374  
              75%                  418.030218  
              max                 4181.844825  
lat           count                 41.000000  
              mean                   9.346293  
              std                    1.965305  
              min                    3.376806  
              25%                    9.107572  
              50%                   10.222383  
              75%                   10.525296  
     

In [43]:
# Describe: recurso solar
solar_cols = [
    "pvout_kwh_kwp_day",
    "annual_yield_kwh_kw_year",
    "s_i_solar",
]

display(describe_by_category(tabla_final, solar_cols))

clasificacion_preliminar        alta_preliminar  baja_preliminar  \
pvout_kwh_kwp_day        count        61.000000       202.000000   
                         mean          4.398787         3.935728   
                         std           0.062449         0.406831   
                         min           4.278000         2.744000   
                         25%           4.363000         3.641000   
                         50%           4.401000         3.955000   
                         75%           4.426000         4.230500   
                         max           4.668000         4.802000   
annual_yield_kwh_kw_year count        61.000000       202.000000   
                         mean       1605.557223      1436.540617   
                         std          22.793765       148.493293   
                         min        1561.469955      1001.559985   
                         25%        1592.494969      1328.965012   
                         50%        1606.365008      1443.574972   
                         75%        1615.490043      1544.132450   
                         max        1703.820081      1752.730017   
s_i_solar                count        61.000000       202.000000   
                         mean          0.785511         0.597123   
                         std           0.025406         0.165513   
                         min           0.736371         0.112286   
                         25%           0.770952         0.477217   
                         50%           0.786412         0.604963   
                         75%           0.796583         0.717046   
                         max           0.895037         0.949552   

clasificacion_preliminar        excluida_preliminar  media_preliminar  \
pvout_kwh_kwp_day        count           699.000000        101.000000   
                         mean              3.833349          4.208653   
                         std               0.470091          0.131812   
                         min               2.468000          3.932000   
                         25%               3.507000          4.120000   
                         50%               3.857000          4.237000   
                         75%               4.185000          4.284000   
                         max               4.926000          4.569000   
annual_yield_kwh_kw_year count           699.000000        101.000000   
                         mean           1399.172412       1536.158507   
                         std             171.583071         48.111334   
                         min             900.819976       1435.179971   
                         25%            1280.054989       1503.799958   
                         50%            1407.805041       1546.504996   
                         75%            1527.525066       1563.659971   
                         max            1797.990043       1667.684915   
s_i_solar                count           699.000000        101.000000   
                         mean              0.555472          0.708158   
                         std               0.191249          0.053626   
                         min               0.000000          0.595606   
                         25%               0.422701          0.672091   
                         50%               0.565094          0.719691   
                         75%               0.698535          0.738812   
                         max               1.000000          0.854760   

clasificacion_preliminar        muy_alta_preliminar  
pvout_kwh_kwp_day        count            41.000000  
                         mean              4.549293  
                         std               0.104045  
                         min               4.416000  
                         25%               4.467000  
                         50%               4.506000  
                         75%               4.602000  
                         max             

In [44]:
# Describe: cercania a red electrica
red_cols = [
    "dist_subestacion_km",
    "g_i_red",
]

display(describe_by_category(tabla_final, red_cols))

clasificacion_preliminar   alta_preliminar  baja_preliminar  \
dist_subestacion_km count        61.000000       202.000000   
                    mean         19.699291        19.650479   
                    std          12.108677        12.101410   
                    min           1.276466         0.843350   
                    25%           9.413902        10.466298   
                    50%          19.568948        16.844272   
                    75%          28.345784        27.730456   
                    max          47.495390        49.578715   
g_i_red             count        61.000000       202.000000   
                    mean          0.973695         0.973761   
                    std           0.016415         0.016405   
                    min           0.936013         0.933189   
                    25%           0.961973         0.962808   
                    50%           0.973872         0.977565   
                    75%           0.987638         0.986212   
                    max           0.998670         0.999257   

clasificacion_preliminar   excluida_preliminar  media_preliminar  \
dist_subestacion_km count           699.000000        101.000000   
                    mean             35.245740         16.921013   
                    std              66.292330         10.924963   
                    min               0.295315          0.745845   
                    25%              12.099908          8.275730   
                    50%              20.221998         15.026998   
                    75%              35.695690         23.022989   
                    max             737.947838         47.484877   
g_i_red             count           699.000000        101.000000   
                    mean              0.952619          0.977461   
                    std               0.089869          0.014810   
                    min               0.000000          0.936027   
                    25%               0.952009          0.969189   
                    50%               0.972986          0.980029   
                    75%               0.983997          0.989181   
                    max               1.000000          0.999389   

clasificacion_preliminar   muy_alta_preliminar  
dist_subestacion_km count            41.000000  
                    mean             14.940180  
                    std              10.902766  
                    min               0.940579  
                    25%               5.902167  
                    50%              12.640689  
                    75%              22.910016  
                    max              39.243127  
g_i_red             count            41.000000  
                    mean              0.980147  
                    std               0.014780  
                    min               0.947200  
                    25%               0.969342  
                    50%               0.983264  
                    75%               0.992399  
                    max               0.999125

In [45]:
# Describe: pendiente y aptitud fisica
pendiente_cols = [
    "p_i_pendiente_proxy",
]

display(describe_by_category(tabla_final, pendiente_cols))
display(pd.crosstab(tabla_final["clasificacion_preliminar"], tabla_final["viabilidad_pendiente"], dropna=False))

clasificacion_preliminar   alta_preliminar  baja_preliminar  \
p_i_pendiente_proxy count             61.0       202.000000   
                    mean               1.0         0.636139   
                    std                0.0         0.223119   
                    min                1.0         0.500000   
                    25%                1.0         0.500000   
                    50%                1.0         0.500000   
                    75%                1.0         1.000000   
                    max                1.0         1.000000   

clasificacion_preliminar   excluida_preliminar  media_preliminar  \
p_i_pendiente_proxy count           697.000000             101.0   
                    mean              0.081779               1.0   
                    std               0.264897               0.0   
                    min               0.000000               1.0   
                    25%               0.000000               1.0   
                    50%               0.000000               1.0   
                    75%               0.000000               1.0   
                    max               1.000000               1.0   

clasificacion_preliminar   muy_alta_preliminar  
p_i_pendiente_proxy count                 41.0  
                    mean                   1.0  
                    std                    0.0  
                    min                    1.0  
                    25%                    1.0  
                    50%                    1.0  
                    75%                    1.0  
                    max                    1.0

viabilidad_pendiente,condicional,desconocida,no_viable,viable
clasificacion_preliminar,,,,
alta_preliminar,0,0,0,61
baja_preliminar,147,0,0,55
excluida_preliminar,14,2,633,50
media_preliminar,0,0,0,101
muy_alta_preliminar,0,0,0,41


In [46]:
# Describe: restricciones ambientales y POT
restriccion_cols = [
    "pct_area_protegida_runap",
    "u_i_no_protegido_runap",
    "u_i_pot_compatible",
    "u_i_uso_suelo",
    "r_i_runap",
    "r_i_zona_urbana_pot",
    "r_i_restriccion_pot",
    "r_i_preliminar",
]

display(describe_by_category(tabla_final, restriccion_cols))
for column in ["tipo_capa_pot", "categoria_aptitud_pot"]:
    if column in tabla_final.columns:
        display(pd.crosstab(tabla_final["clasificacion_preliminar"], tabla_final[column], dropna=False))

clasificacion_preliminar        alta_preliminar  baja_preliminar  \
pct_area_protegida_runap count        61.000000       202.000000   
                         mean          0.018319         0.106289   
                         std           0.062813         0.169227   
                         min           0.000000         0.000000   
                         25%           0.000000         0.000000   
...                                         ...              ...   
r_i_preliminar           min           1.000000         1.000000   
                         25%           1.000000         1.000000   
                         50%           1.000000         1.000000   
                         75%           1.000000         1.000000   
                         max           1.000000         1.000000   

clasificacion_preliminar        excluida_preliminar  media_preliminar  \
pct_area_protegida_runap count           699.000000        101.000000   
                         mean              0.139512          0.056953   
                         std               0.210681          0.102760   
                         min               0.000000          0.000000   
                         25%               0.000000          0.000000   
...                                             ...               ...   
r_i_preliminar           min               0.000000          1.000000   
                         25%               0.000000          1.000000   
                         50%               0.000000          1.000000   
                         75%               0.000000          1.000000   
                         max               0.000000          1.000000   

clasificacion_preliminar        muy_alta_preliminar  
pct_area_protegida_runap count            41.000000  
                         mean              0.014841  
                         std               0.035522  
                         min               0.000000  
                         25%               0.000000  
...                                             ...  
r_i_preliminar           min               1.000000  
                         25%               1.000000  
                         50%               1.000000  
                         75%               1.000000  
                         max               1.000000  

[64 rows x 5 columns]

tipo_capa_pot,rural,NaN
clasificacion_preliminar,,
alta_preliminar,0,61
baja_preliminar,0,202
excluida_preliminar,3,696
media_preliminar,0,101
muy_alta_preliminar,0,41


categoria_aptitud_pot,condicional_forestal,restriccion_ambiental,sin_datos,NaN
clasificacion_preliminar,,,,
alta_preliminar,0,0,0,61
baja_preliminar,0,0,9,193
excluida_preliminar,1,2,61,635
media_preliminar,0,0,2,99
muy_alta_preliminar,0,0,0,41


In [47]:
# Describe: demanda XM como contexto, no como criterio principal del ranking rural
demanda_cols = [
    "demanda_xm_proxy_mwh_o_unidad_fuente",
    "d_i_demanda",
    "bono_demanda_favorable",
    "flag_atipico_eda_demanda",
    "flag_revision_demanda",
]

display(describe_by_category(tabla_final, demanda_cols))
for column in ["zona_xm_demanda", "tipo_mapeo_demanda"]:
    if column in tabla_final.columns:
        display(pd.crosstab(tabla_final["clasificacion_preliminar"], tabla_final[column], dropna=False))

clasificacion_preliminar                    alta_preliminar  baja_preliminar  \
demanda_xm_proxy_mwh_o_unidad_fuente count        61.000000       192.000000   
                                     mean        573.764671       694.336742   
                                     std         270.994579       600.174216   
                                     min         216.023961        23.203969   
                                     25%         367.461738       292.716300   
                                     50%         517.118784       416.858178   
                                     75%         740.403096      1136.901610   
                                     max        1974.450982      1974.450982   
d_i_demanda                          count        61.000000       192.000000   
                                     mean          0.282158         0.343951   
                                     std           0.138883         0.307585   
                                     min           0.098819         0.000000   
                                     25%           0.176430         0.138123   
                                     50%           0.253128         0.201745   
                                     75%           0.367559         0.570762   
                                     max           1.000000         1.000000   
bono_demanda_favorable               count        61.000000       202.000000   
                                     mean          0.014108         0.016346   
                                     std           0.006944         0.015451   
                                     min           0.004941         0.000000   
                                     25%           0.008821         0.006607   
                                     50%           0.012656         0.010087   
                                     75%           0.018378         0.019302   
                                     max           0.050000         0.050000   
flag_atipico_eda_demanda             count        61.000000       202.000000   
                                     mean          0.377049         0.252475   
                                     std           0.488669         0.435512   
                                     min           0.000000         0.000000   
                                     25%           0.000000         0.000000   
                                     50%           0.000000         0.000000   
                                     75%           1.000000         0.750000   
                                     max           1.000000         1.000000   
flag_revision_demanda                count        61.000000       202.000000   
                                     mean          0.377049         0.252475   
                                     std           0.488669         0.435512   
                                     min           0.000000         0.000000   
                                     25%           0.000000         0.000000   
                                     50%           0.000000         0.000000   
                                     75%           1.000000         0.750000   
                                     max           1.000000         1.000000   

clasificacion_preliminar                    excluida_preliminar  \
demanda_xm_proxy_mwh_o_unidad_fuente count           661.000000   
                                     mean            652.232750   
                                     std             571.693445   
                                     min              23.203969   
                                     25%             281.046130   
                                     50%             367.461738   
                                     75%            1136.901610   
                                     max            1974.450982   
d_i_demanda                          count           661.000000   
                                     mean        

zona_xm_demanda,SubAntioquia,SubArauca,SubAtlantico,SubBogota,SubBolivar,SubBoyaca-Casanare,SubCQR,SubCaqueta,SubCauca-Narinno,SubCordoba-Sucre,SubGCM,SubHuila-Tolima,SubMeta,SubNorteSantander,SubPutumayo,SubSantander,SubValle,NaN
clasificacion_preliminar,,,,,,,,,,,,,,,,,,
alta_preliminar,1,0,8,1,10,1,5,0,3,5,17,7,0,1,0,1,1,0
baja_preliminar,21,0,2,28,5,23,8,3,17,21,6,17,7,6,5,15,8,10
excluida_preliminar,99,7,4,76,13,106,39,13,83,8,21,47,15,31,8,63,28,38
media_preliminar,3,0,1,12,10,11,1,0,3,22,9,13,7,2,0,4,3,0
muy_alta_preliminar,1,0,8,0,8,1,0,0,0,0,17,0,0,0,0,4,2,0


tipo_mapeo_demanda,ambigua_cundinamarca_subbogota,compuesto_acronimo_inferido,compuesto_explicito,directo_departamento,especial_bogota_directo,sin_mapeo_xm
clasificacion_preliminar,,,,,,
alta_preliminar,1,22,16,22,0,0
baja_preliminar,27,14,78,72,1,10
excluida_preliminar,76,60,244,281,0,38
media_preliminar,12,10,49,30,0,0
muy_alta_preliminar,0,17,1,23,0,0


In [48]:
# Describe: puntajes finales de viabilidad
score_cols = [
    "v_i_modelo_rural",
    "score_rural_con_bono_demanda",
    "v_i_modelo_proxy_xm",
    "v_i_modelo_oficial",
    "score_preliminar_solar_red_pendiente_runap",
]

display(describe_by_category(tabla_final, score_cols))
display(
    tabla_final.sort_values("v_i_modelo_rural", ascending=False)[
        existing_columns(tabla_final, [
            "codigo_dane", "municipio", "departamento", "clasificacion_preliminar",
            "v_i_modelo_rural", "pvout_kwh_kwp_day", "dist_subestacion_km",
            "cluster_kmeans_label"
        ])
    ].head(20)
)

clasificacion_preliminar                          alta_preliminar  \
v_i_modelo_rural                           count        61.000000   
                                           mean          0.915206   
                                           std           0.005637   
                                           min           0.905196   
                                           25%           0.910823   
                                           50%           0.915403   
                                           75%           0.920116   
                                           max           0.923824   
score_rural_con_bono_demanda               count        61.000000   
                                           mean          0.929313   
                                           std           0.009209   
                                           min           0.912416   
                                           25%           0.923325   
                                           50%           0.928190   
                                           75%           0.936261   
                                           max           0.961324   
v_i_modelo_proxy_xm                        count        61.000000   
                                           mean          0.915206   
                                           std           0.005637   
                                           min           0.905196   
                                           25%           0.910823   
                                           50%           0.915403   
                                           75%           0.920116   
                                           max           0.923824   
v_i_modelo_oficial                         count        61.000000   
                                           mean          0.915206   
                                           std           0.005637   
                                           min           0.905196   
                                           25%           0.910823   
                                           50%           0.915403   
                                           75%           0.920116   
                                           max           0.923824   
score_preliminar_solar_red_pendiente_runap count        61.000000   
                                           mean          0.915206   
                                           std           0.005637   
                                           min           0.905196   
                                           25%           0.910823   
                                           50%           0.915403   
                                           75%           0.920116   
                                           max           0.923824   

clasificacion_preliminar                          baja_preliminar  \
v_i_modelo_rural                           count       202.000000   
                                           mean          0.749527   
                                           std           0.072505   
                                           min           0.545524   
                                           25%           0.709255   
                                           50%           0.756510   
                                           75%           0.798261   
                                           max           0.855838   
score_rural_con_bono_demanda               count       202.000000   
                                           mean          0.765873   
                                           std           0.073528   
                                           min           0.550465   
                                           25%           0.721136   
                                           50%           0.770662   
                                           75%           0.819287   
                                           max           0.902144 

,codigo_dane,municipio,departamento,clasificacion_preliminar,v_i_modelo_rural,pvout_kwh_kwp_day,dist_subestacion_km,cluster_kmeans_label
855,68121,Cabrera,Santander,muy_alta_preliminar,0.974961,4.789,11.667284,cluster_0
851,68079,Barichara,Santander,muy_alta_preliminar,0.972228,4.772,12.921475,cluster_0
901,68500,Oiba,Santander,muy_alta_preliminar,0.971996,4.739,3.681041,cluster_0
425,20750,San Diego,Cesar,muy_alta_preliminar,0.959974,4.686,14.684426,cluster_0
125,8001,Barranquilla,Atlántico,muy_alta_preliminar,0.955547,4.621,2.810665,cluster_0
675,47570,Puebloviejo,Magdalena,muy_alta_preliminar,0.955469,4.710,24.597827,cluster_0
277,15664,San José De Pare,Boyacá,muy_alta_preliminar,0.955400,4.639,9.474433,cluster_0
413,20250,El Paso,Cesar,muy_alta_preliminar,0.952432,4.599,2.345905,cluster_0
144,8758,Soledad,Atlántico,muy_alta_preliminar,0.950238,4.584,2.911517,cluster_0
403,20001,Valledupar,Cesar,muy_alta_preliminar,0.949905,4.804,36.336098,cluster_0


In [49]:
# Exportar CSV: ranking de los 500 municipios mas probables con todas sus caracteristicas
ranking_output_dir = PROJECT_ROOT / "data" / "clean" / "rankings_exportados"
ranking_output_dir.mkdir(parents=True, exist_ok=True)

score_col = "v_i_modelo_rural" if "v_i_modelo_rural" in tabla_final.columns else "v_i_modelo_proxy_xm"
ranking_500 = (
    tabla_final.copy()
    .assign(ranking_score=pd.to_numeric(tabla_final[score_col], errors="coerce"))
    .sort_values("ranking_score", ascending=False)
    .head(500)
    .reset_index(drop=True)
)
ranking_500.insert(0, "ranking", range(1, len(ranking_500) + 1))

ranking_path = ranking_output_dir / "ranking_top_500_municipios_probables.csv"
ranking_500.to_csv(ranking_path, index=False, encoding="utf-8-sig")

print(f"CSV exportado: {ranking_path}")
print(f"Filas exportadas: {len(ranking_500):,}")
print(f"Columnas exportadas: {len(ranking_500.columns):,}")
display(ranking_500.head(20))

CSV exportado: c:\Users\tabo_\OneDrive\Desktop\JHON T\ITM\introduccion inteligencia artificial\data\clean\rankings_exportados\ranking_top_500_municipios_probables.csv
Filas exportadas: 500
Columnas exportadas: 79


,ranking,codigo_dane,municipio,departamento,categoria_nombre,area_km2_igac,altitud_m,lon,lat,pvout_kwh_kwp_day,...,score_rural_con_bono_demanda,v_i_modelo_proxy_xm,clasificacion_preliminar,v_i_modelo_oficial,estado_modelo_oficial,notas_metodologicas,cluster_kmeans,cluster_kmeans_label,silhouette_municipio,ranking_score
0,1,68121,Cabrera,Santander,Municipio,65.457118,1020,-73.253978,6.569504,4.789,...,0.985049,0.974961,muy_alta_preliminar,0.974961,modelo_rural_sin_demanda_con_pot_pendiente: no...,"Score preliminar usa PVOUT puntual municipal, ...",0.0,cluster_0,0.588490,0.974961
1,2,68079,Barichara,Santander,Municipio,137.027360,1284,-73.226954,6.646217,4.772,...,0.982316,0.972228,muy_alta_preliminar,0.972228,modelo_rural_sin_demanda_con_pot_pendiente: no...,"Score preliminar usa PVOUT puntual municipal, ...",0.0,cluster_0,0.596236,0.972228
2,3,68500,Oiba,Santander,Municipio,287.340765,1420,-73.276476,6.230834,4.739,...,0.982083,0.971996,muy_alta_preliminar,0.971996,modelo_rural_sin_demanda_con_pot_pendiente: no...,"Score preliminar usa PVOUT puntual municipal, ...",0.0,cluster_0,0.609154,0.971996
3,4,20750,San Diego,Cesar,Municipio,644.881803,167,-73.355465,10.111936,4.686,...,0.978352,0.959974,muy_alta_preliminar,0.959974,modelo_rural_sin_demanda_con_pot_pendiente: no...,"Score preliminar usa PVOUT puntual municipal, ...",0.0,cluster_0,0.635525,0.959974
4,5,8001,Barranquilla,Atlántico,Distrito,153.791295,32,-74.823549,11.009739,4.621,...,0.972822,0.955547,muy_alta_preliminar,0.955547,modelo_rural_sin_demanda_con_pot_pendiente: no...,"Score preliminar usa PVOUT puntual municipal, ...",0.0,cluster_0,0.663986,0.955547
5,6,47570,Puebloviejo,Magdalena,Municipio,679.110538,1,-74.369513,10.831889,4.710,...,0.973847,0.955469,muy_alta_preliminar,0.955469,modelo_rural_sin_demanda_con_pot_pendiente: no...,"Score preliminar usa PVOUT puntual municipal, ...",0.0,cluster_0,0.625552,0.955469
6,7,15664,San José De Pare,Boyacá,Municipio,74.284809,1525,-73.536645,6.000596,4.639,...,0.962406,0.955400,muy_alta_preliminar,0.955400,modelo_rural_sin_demanda_con_pot_pendiente: no...,"Score preliminar usa PVOUT puntual municipal, ...",0.0,cluster_0,0.657222,0.955400
7,8,20250,El Paso,Cesar,Municipio,812.313879,37,-73.722012,9.704085,4.599,...,0.970810,0.952432,muy_alta_preliminar,0.952432,modelo_rural_sin_demanda_con_pot_pendiente: no...,"Score preliminar usa PVOUT puntual municipal, ...",0.0,cluster_0,0.674346,0.952432
8,9,8758,Soledad,Atlántico,Municipio,59.127993,14,-74.773649,10.910411,4.584,...,0.967513,0.950238,muy_alta_preliminar,0.950238,modelo_rural_sin_demanda_con_pot_pendiente: no...,"Score preliminar usa PVOUT puntual municipal, ...",0.0,cluster_0,0.680925,0.950238
9,10,20001,Valledupar,Cesar,Municipio,4181.844825,169,-73.366821,10.343420,4.804,...,0.968283,0.949905,muy_alta_preliminar,0.949905,modelo_rural_sin_demanda_con_pot_pendiente: no...,"Score preliminar usa PVOUT puntual municipal, ...",0.0,cluster_0,0.525755,0.949905


In [50]:
# Describe: categorias por cluster K-Means
cluster_cols = [
    "v_i_modelo_rural",
    "pvout_kwh_kwp_day",
    "dist_subestacion_km",
    "s_i_solar",
    "g_i_red",
    "p_i_pendiente_proxy",
    "u_i_uso_suelo",
    "silhouette_municipio",
]

cluster_work = tabla_final[existing_columns(tabla_final, ["cluster_kmeans_label", *cluster_cols])].copy()
for column in [column for column in cluster_work.columns if column != "cluster_kmeans_label"]:
    cluster_work[column] = pd.to_numeric(cluster_work[column], errors="coerce")

display(cluster_work.groupby("cluster_kmeans_label", dropna=False).describe().T)
display(pd.crosstab(tabla_final["cluster_kmeans_label"], tabla_final["clasificacion_preliminar"], dropna=False))

cluster_kmeans_label         cluster_0   cluster_1    NaN
v_i_modelo_rural     count  258.000000  147.000000  699.0
                     mean     0.885840    0.725213    0.0
                     std      0.046801    0.066976    0.0
                     min      0.706574    0.545524    0.0
                     25%      0.860563    0.681697    0.0
...                                ...         ...    ...
silhouette_municipio min      0.126884    0.243368    NaN
                     25%      0.628831    0.474498    NaN
                     50%      0.724539    0.562010    NaN
                     75%      0.752848    0.612673    NaN
                     max      0.766394    0.633783    NaN

[64 rows x 3 columns]

clasificacion_preliminar,alta_preliminar,baja_preliminar,excluida_preliminar,media_preliminar,muy_alta_preliminar
cluster_kmeans_label,,,,,
cluster_0,61,55,0,101,41
cluster_1,0,147,0,0,0
NaN,0,0,699,0,0


In [51]:
# Describe rapido de tablas finales cargadas en SQL
tablas_sql = [
    "viabilidad_municipal",
    "cluster_municipal",
    "cluster_perfiles",
    "cluster_centroides",
    "cluster_evaluacion_k",
    "cluster_resumen_seleccion",
    "solar_escenarios",
    "precios_energia_departamento",
]

for tabla in tablas_sql:
    print(f"\n===== {tabla} =====")
    df_sql = sql_df(f"SELECT * FROM `{tabla}`")
    print(f"Filas: {len(df_sql):,} | Columnas: {len(df_sql.columns):,}")
    display(df_sql.head())
    display(df_sql.describe(include="all").T)


===== viabilidad_municipal =====
Filas: 1,104 | Columnas: 74


,codigo_dane,municipio,departamento,categoria_nombre,area_km2_igac,altitud_m,lon,lat,pvout_kwh_kwp_day,annual_yield_kwh_kw_year,...,r_i_preliminar,v_i_modelo_rural,score_preliminar_solar_red_pendiente_runap,bono_demanda_favorable,score_rural_con_bono_demanda,v_i_modelo_proxy_xm,clasificacion_preliminar,v_i_modelo_oficial,estado_modelo_oficial,notas_metodologicas
0,5001,Medellín,Antioquia,Distrito,373.440416,1475,-75.602895,6.269145,4.371,1595.414932,...,0,0.0,0.0,0.0,0.0,0.0,excluida_preliminar,0.0,modelo_rural_sin_demanda_con_pot_pendiente: no...,"Score preliminar usa PVOUT puntual municipal, ..."
1,5002,Abejorral,Antioquia,Municipio,506.952798,2275,-75.429630,5.805339,4.286,1564.389918,...,0,0.0,0.0,0.0,0.0,0.0,excluida_preliminar,0.0,modelo_rural_sin_demanda_con_pot_pendiente: no...,"Score preliminar usa PVOUT puntual municipal, ..."
2,5004,Abriaquí,Antioquia,Municipio,296.974192,1900,-76.083417,6.629176,3.780,1379.699990,...,0,0.0,0.0,0.0,0.0,0.0,excluida_preliminar,0.0,modelo_rural_sin_demanda_con_pot_pendiente: no...,"Score preliminar usa PVOUT puntual municipal, ..."
3,5021,Alejandría,Antioquia,Municipio,128.856440,1750,-75.099281,6.356241,4.103,1497.595060,...,0,0.0,0.0,0.0,0.0,0.0,excluida_preliminar,0.0,modelo_rural_sin_demanda_con_pot_pendiente: no...,"Score preliminar usa PVOUT puntual municipal, ..."
4,5030,Amagá,Antioquia,Municipio,84.118977,1400,-75.703795,6.034324,4.134,1508.909936,...,0,0.0,0.0,0.0,0.0,0.0,excluida_preliminar,0.0,modelo_rural_sin_demanda_con_pot_pendiente: no...,"Score preliminar usa PVOUT puntual municipal, ..."


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
codigo_dane,1104.0,NaN,NaN,NaN,37759.651268,25782.638439,5001.0,15656.5,25789.0,63418.25,99773.0
municipio,1104,1020,La Unión,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
departamento,1104,32,Antioquia,125,NaN,NaN,NaN,NaN,NaN,NaN,NaN
categoria_nombre,1104,2,Municipio,1092,NaN,NaN,NaN,NaN,NaN,NaN,NaN
area_km2_igac,1104.0,NaN,NaN,NaN,879.538032,2978.924623,15.732852,130.882807,279.439405,642.022539,65187.505271
...,...,...,...,...,...,...,...,...,...,...,...
v_i_modelo_proxy_xm,1104.0,NaN,NaN,NaN,0.303581,0.403116,0.0,0.0,0.0,0.778236,0.974961
clasificacion_preliminar,1104,5,excluida_preliminar,699,NaN,NaN,NaN,NaN,NaN,NaN,NaN
v_i_modelo_oficial,1104.0,NaN,NaN,NaN,0.303581,0.403116,0.0,0.0,0.0,0.778236,0.974961
estado_modelo_oficial,1104,4,modelo_rural_sin_demanda_con_pot_pendiente: no...,816,NaN,NaN,NaN,NaN,NaN,NaN,NaN



===== cluster_municipal =====
Filas: 405 | Columnas: 88


,codigo_dane,municipio,departamento,categoria_nombre,area_km2_igac,altitud_m,lon,lat,pvout_kwh_kwp_day,annual_yield_kwh_kw_year,...,g_i_red_valor_imputacion_kmeans,p_i_pendiente_proxy_imputado_kmeans,p_i_pendiente_proxy_valor_imputacion_kmeans,u_i_uso_suelo_imputado_kmeans,u_i_uso_suelo_valor_imputacion_kmeans,cluster_kmeans,cluster_kmeans_label,distancia_centroide_kmeans,silhouette_municipio,silhouette_modelo_k
0,5038,Angostura,Antioquia,Municipio,338.354474,1675,-75.353143,6.869306,4.146,1513.289967,...,0.978506,0,1,0,0.998306,1,cluster_1,0.112102,0.619518,0.618038
1,5079,Barbosa,Antioquia,Municipio,205.603285,1305,-75.336878,6.439327,4.240,1547.599916,...,0.978506,0,1,0,0.998306,0,cluster_0,0.061574,0.755289,0.618038
2,5120,Cáceres,Antioquia,Municipio,1872.696760,114,-75.241323,7.654024,4.138,1510.370004,...,0.978506,0,1,0,0.998306,0,cluster_0,0.161967,0.591930,0.618038
3,5147,Carepa,Antioquia,Municipio,387.435334,34,-76.748445,7.798095,3.839,1401.234995,...,0.978506,0,1,0,0.998306,0,cluster_0,0.164526,0.619846,0.618038
4,5154,Caucasia,Antioquia,Municipio,1428.239321,51,-75.048746,7.837624,4.131,1507.815015,...,0.978506,0,1,0,0.998306,1,cluster_1,0.101203,0.625097,0.618038


...

,scenario_name,source_capex,source_land_use,source_yield,capex_usd_per_kw,land_use_hectares_per_mw,annual_yield_kwh_per_kw_year,capacidad_kw_por_hectarea,costo_usd_por_hectarea,generacion_kwh_por_hectarea_anual,...,costo_usd_por_mwh_anual_simple,exchange_rate_cop_per_usd,costo_cop_por_hectarea,notas_metodologicas,source_capex_url,source_land_use_url,source_yield_url,dato_fuente_original,supuesto_adoptado,formula_yield
0,base,IRENA Renewable Power Generation Costs in 2024...,"NREL land-use report, large PV total area aver...","World Bank/ESMAP Global Solar Atlas, Colombia ...",599,3.197017,1477.958000,312.791622,187362.181644,462292.880231,...,405.288919,NaN,NaN,PVOUT es promedio pais; para scoring final deb...,https://www.irena.org/-/media/Files/IRENA/Agen...,https://docs.nrel.gov/docs/fy13osti/56290.pdf,https://datacatalog.worldbank.org/search/datas...,CAPEX: 599 USD/kW en proyeccion IRENA; uso sue...,Escenario medio: costo proyectado de corto pla...,annual_yield = PVOUT_diario_Colombia * 365
1,conservador,IRENA Renewable Power Generation Costs in 2024...,"NREL land-use report, large PV 1-axis total ar...","UPME/IDEAM Atlas, Costa Pacifica 1,278 kWh/m2-...",691,3.358891,1063.258188,297.717327,205722.673005,316550.385655,...,649.889188,NaN,NaN,No representa diseno de ingenieria; usa total ...,https://www.irena.org/-/media/Files/IRENA/Agen...,https://docs.nrel.gov/docs/fy13osti/56290.pdf,https://www1.upme.gov.co/Hemeroteca/Impresos/A...,CAPEX: 691 USD/kW; uso suelo: 8.3 acres/MWac; ...,Escenario de menor productividad: mayor uso de...,annual_yield = GHI_regional_anual * (PVOUT_pro...
2,optimista,IRENA Renewable Power Generation Costs in 2024...,"NREL land-use report, large PV fixed total are...","UPME/IDEAM Atlas, Guajira 2,190 kWh/m2-year; a...",534,3.035142,1822.015204,329.473842,175939.031605,600306.349509,...,293.082077,NaN,NaN,El resultado depende fuertemente de localizar ...,https://www.irena.org/-/media/Files/IRENA/Agen...,https://docs.nrel.gov/docs/fy13osti/56290.pdf,https://www1.upme.gov.co/Hemeroteca/Impresos/A...,CAPEX: 534 USD/kW en proyeccion IRENA; uso sue...,Escenario de mayor productividad: menor uso de...,annual_yield = GHI_regional_anual * (PVOUT_pro...


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
scenario_name,3,3,base,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
source_capex,3,3,IRENA Renewable Power Generation Costs in 2024...,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
source_land_use,3,3,"NREL land-use report, large PV total area aver...",1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
source_yield,3,3,"World Bank/ESMAP Global Solar Atlas, Colombia ...",1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
capex_usd_per_kw,3.0,NaN,NaN,NaN,608.0,78.885994,534.0,566.5,599.0,645.0,691.0
land_use_hectares_per_mw,3.0,NaN,NaN,NaN,3.197017,0.161874,3.035142,3.116079,3.197017,3.277954,3.358891
annual_yield_kwh_per_kw_year,3.0,NaN,NaN,NaN,1454.410464,379.9262,1063.258188,1270.608094,1477.958,1649.986602,1822.015204
capacidad_kw_por_hectarea,3.0,NaN,NaN,NaN,313.327597,15.88504,297.717327,305.254475,312.791622,321.132732,329.473842
costo_usd_por_hectarea,3.0,NaN,NaN,NaN,189674.628751,15025.873768,175939.031605,181650.606624,187362.181644,196542.427324,205722.673005
generacion_kwh_por_hectarea_anual,3.0,NaN,NaN,NaN,459716.538465,141895.524624,316550.385655,389421.632943,462292.880231,531299.61487,600306.349509



===== precios_energia_departamento =====
Filas: 7 | Columnas: 9


,precio_id,departamento_id,departamento,departamento_normalizado,prestador_tarifa,precio_compra_cop_kwh,fecha_publicacion,fuente_url,nota
0,1,4,Atlantico,ATLANTICO,Air-e,796,2026-01-09,https://www.minenergia.gov.co/es/sala-de-prens...,Promedio oficial Minenergia para departamentos...
1,2,18,La Guajira,LA GUAJIRA,Air-e,796,2026-01-09,https://www.minenergia.gov.co/es/sala-de-prens...,Promedio oficial Minenergia para departamentos...
2,3,19,Magdalena,MAGDALENA,Air-e,796,2026-01-09,https://www.minenergia.gov.co/es/sala-de-prens...,Promedio oficial Minenergia para departamentos...
3,4,5,Bolivar,BOLIVAR,AFINIA,879,2026-01-09,https://www.minenergia.gov.co/es/sala-de-prens...,Promedio oficial Minenergia para departamentos...
4,5,11,Cesar,CESAR,AFINIA,879,2026-01-09,https://www.minenergia.gov.co/es/sala-de-prens...,Promedio oficial Minenergia para departamentos...


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
precio_id,7.0,NaN,NaN,NaN,4.0,2.160247,1.0,2.5,4.0,5.5,7.0
departamento_id,7.0,NaN,NaN,NaN,14.0,8.445906,4.0,8.0,13.0,18.5,28.0
departamento,7,7,Atlantico,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
departamento_normalizado,7,7,ATLANTICO,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
prestador_tarifa,7,2,AFINIA,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
precio_compra_cop_kwh,7.0,NaN,NaN,NaN,843.428571,44.365366,796.0,796.0,879.0,879.0,879.0
fecha_publicacion,7,1,2026-01-09,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN
fuente_url,7,1,https://www.minenergia.gov.co/es/sala-de-prens...,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN
nota,7,2,Promedio oficial Minenergia para departamentos...,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
